<a href="https://colab.research.google.com/github/JoBeaMi/CAIDI-app/blob/main/IDEAL_Notebook1_Whisper_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IDEAL — Fase 1: Transcrição Automática (faster-whisper)
## Instrumento Digital de Elicitação de Amostras de Linguagem
**ESS-IPS / CAIDI — v0.3 Março 2026**

---

## Pipeline IDEAL — visão geral

```
📓 NOTEBOOK 1 (este)          📓 NOTEBOOK 2                  📓 NOTEBOOK 3
faster-whisper                Stanza + LARSP-PE              Emissor .cha final
─────────────────             ─────────────────              ────────────────────
Áudio → .cha base             .cha anotado →                 Excel revisto →
      → .txt revisão          Excel LARSP-PE                 .cha final com
      → .srt para VLC         (sugestões azul/amarelo)       %syn: %mor: %err: %lar:
         ↓                           ↓                              ↓
   ✏️ REVISÃO MANUAL           ✏️ REVISÃO EXCEL               ✅ arquivo IDEAL
   (CHAT: [/] [//] (.) &-hm    (confirmar/corrigir            (compatível CLAN)
    [* m:conc]...)              funções sint. e sintagmas)
```

**Porquê três notebooks separados?**\
O faster-whisper transcreve mas comete erros — a revisão humana é obrigatória antes da análise linguística.\
O Stanza analisa mas não é perfeito em dados clínicos — especialmente em funções sintáticas (117 combinações) e sintagmas (150+). A revisão do Excel é obrigatória antes de emitir o .cha final.\
O `.cha` final reflecte sempre a análise humana corrigida, não a análise automática bruta.

---

## O que este notebook faz (Fase 1)

Transcreve o áudio com **faster-whisper large-v3** e gera três ficheiros:
- **`.cha` base** — formato CHAT/CHILDES pré-formatado, pronto para revisão
- **`.txt` de revisão** — mesma transcrição com instruções de anotação inline
- **`.srt`** — legendas para sincronizar com VLC enquanto se revê

### faster-whisper vs openai-whisper
| | openai-whisper | **faster-whisper (IDEAL)** |
|---|---|---|
| Motor | PyTorch | CTranslate2 (optimizado) |
| Velocidade | baseline | ✅ 2-4x mais rápido |
| Qualidade de transcrição | igual | igual |
| Timestamps por palavra | ✅ | ✅ |
| Pausas inline `(.)` `(..)` `(...)` | ✅ | ✅ |
| Instalação no Colab | ✅ | ✅ estável |

O faster-whisper é o motor de transcrição Whisper optimizado com CTranslate2 — mesma qualidade, 2-4x mais rápido, sem conflitos de dependências no Colab.

### O que é gerado automaticamente:
| Elemento | Automático | Notas |
|---|---|---|
| Cabeçalho `@Begin`...`@End` | ✅ | com metadados do participante |
| `*CHI:` + `[MM:SS]` | ✅ | prefixo e timestamp por enunciado |
| `(.)` `(..)` `(...)` no texto | ✅ | via word_timestamps |
| `⚡` suspeita de supressão | ✅ | gap entre palavras consecutivas |
| `🔴` baixa confiança | ✅ | avg_logprob baixo |
| `%com:` duração + alertas | ✅ | linha dependente automática |

### O que a aluna preenche manualmente:
| Elemento | Descrição |
|---|---|
| `[/]` `[//]` `[///]` | repetições, revisões, reformulações |
| `&-hm` `&-ã` | preenchimentos de pausa |
| `[* m:conc]` `[* m:conc_num]` `[* m:conc_gen]` | erros de concordância |
| `[* m:cv]` | erro de concordância verbal |
| `[* m:prep]` `[* m:art]` | erros de preposição / artigo |
| `[?]` | palavra ininteligível |
| `M:` | linha não analisável |
| Correcções de transcrição | especialmente nos locais ⚡ e 🔴 |

---

## ⚠️ Passo 0 — Activar GPU
**Editar → Configurações do notebook → Acelerador de hardware → GPU (T4) → Guardar**


In [1]:
# ============================================================
# CÉLULA 1 — Instalação faster-whisper
# IDEAL – ESS-IPS / CAIDI — v0.3 Março 2026
# Correr UMA VEZ por sessão (~2 min)
# ============================================================

print('A instalar faster-whisper...')
!pip install -q faster-whisper
!apt-get install -q -y ffmpeg
print('\n✅ Instalação concluída!')

A instalar faster-whisper...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 54.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.

✅ Instalação concluída!


In [2]:
# ============================================================
# CÉLULA 2 — Carregar modelo faster-whisper
# Correr UMA VEZ por sessão (~2 min)
# ============================================================

import torch
from faster_whisper import WhisperModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'

if device == 'cuda':
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   compute_type: {compute_type}')
else:
    print('⚠️  Sem GPU — activa nas configurações do notebook para maior velocidade')

print('\nA carregar faster-whisper large-v3...')
model = WhisperModel('large-v3', device=device, compute_type=compute_type)
print('✅ Modelo pronto!')
print('\n   faster-whisper large-v3 — mesma qualidade, mais rápido.')
print('   Timestamps por palavra disponíveis nativamente.')

✅ GPU: Tesla T4
   compute_type: float16

A carregar faster-whisper large-v3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Modelo pronto!

   faster-whisper large-v3 — mesma qualidade, mais rápido.
   Timestamps por palavra disponíveis nativamente.


In [3]:
# ============================================================
# CÉLULA 3 — Funções de formatação CHAT
# Correr UMA VEZ por sessão
# ============================================================

import re
import datetime
import os

def ts(segundos):
    """Converte segundos para [MM:SS]"""
    m = int(segundos) // 60
    s = int(segundos) % 60
    return f'[{m:02d}:{s:02d}]'

def ts_srt(segundos):
    """Converte segundos para formato SRT HH:MM:SS,mmm"""
    ms = int(segundos * 1000)
    h = ms // 3600000
    m = (ms % 3600000) // 60000
    s = (ms % 60000) // 1000
    mss = ms % 1000
    return f'{h:02d}:{m:02d}:{s:02d},{mss:03d}'

def limpar_texto(texto):
    """Limpeza básica do texto Whisper para CHAT."""
    t = texto.strip()
    # Remover pontuação dupla
    t = re.sub(r'[.!?]+$', '', t).strip()
    # Minúsculas (CHAT usa maiúscula só para nomes próprios)
    if t:
        t = t[0].lower() + t[1:]
    return t

def detectar_pausa_longa(seg_actual, seg_seguinte, limiar_curta=0.5, limiar_media=1.0, limiar_longa=2.0):
    """Detecta pausa entre segmentos e classifica segundo limiares configuráveis."""
    if seg_seguinte is None:
        return None
    gap = seg_seguinte['start'] - seg_actual['end']
    if gap >= limiar_longa:
        return '(...)'
    elif gap >= limiar_media:
        return '(..)'
    elif gap >= limiar_curta:
        return '(.)'
    return None

def gerar_cha(resultado, codigo, grupo, idade, historia, tarefa,
               avaliadora, data, duracao_min,
               limiar_curta=0.5, limiar_media=1.0, limiar_longa=2.0):
    """Gera ficheiro .cha (CHAT) com máximo de formatação automática."""
    linhas = []
    segs = resultado['segments']

    # ── Cabeçalho CHAT ────────────────────────────────────────
    linhas.append('@Begin')
    linhas.append('@Languages:\tpt')
    linhas.append(f'@Participants:\tCHI {codigo} Target_Child, INV Investigadora Investigator')
    linhas.append(f'@ID:\tpt|IDEAL|CHI|{idade}||{grupo}||Target_Child|')
    linhas.append(f'@ID:\tpt|IDEAL|INV||||Investigator||')
    linhas.append(f'@Date:\t{data}')
    linhas.append(f'@Comment:\tHistoria={historia} Tarefa={tarefa} Avaliadora={avaliadora}')
    linhas.append(f'@Comment:\tDuracao={duracao_min}min Modelo=Whisper-large-v3')
    linhas.append(f'@Comment:\t⚠️ Transcrição automática — rever e completar anotações CHAT')
    linhas.append('@Comment:\tConvenções: [/]=repetição [//]=revisão (.)=pausa &-ã=preenchimento')
    linhas.append('@Comment:\t[* m:conc]=erro concordância [* m:prep]=erro preposição [?]=ininteligível')
    linhas.append('@Comment:\tM: no início da linha = enunciado não analisável (maze)')
    linhas.append('')

    # ── Enunciados ────────────────────────────────────────────
    for i, seg in enumerate(segs):
        texto_base = limpar_texto(seg['text'])
        if not texto_base:
            continue

        timestamp = ts(seg['start'])
        seg_seguinte = segs[i+1] if i+1 < len(segs) else None

        # ── Inserir pausas inline entre palavras ──────────────
        # Usa word_timestamps para inserir (.) (..) (...) no sítio certo
        palavras_seg = seg.get('words', [])
        if palavras_seg:
            partes = []
            for j, word in enumerate(palavras_seg):
                partes.append(word['word'])
                if j < len(palavras_seg) - 1:
                    gap = palavras_seg[j+1]['start'] - word['end']
                    if gap >= limiar_longa:
                        partes.append('(...)')
                    elif gap >= limiar_media:
                        partes.append('(..)')
                    elif gap >= limiar_curta:
                        partes.append('(.)')
            texto = ' '.join(partes).strip()
            # Normalizar espaços duplos
            import re as _re
            texto = _re.sub(r' +', ' ', texto)
            texto = texto[0].lower() + texto[1:] if texto else texto
        else:
            texto = texto_base

        # Flag de baixa confiança no início da linha
        prefixo = ''
        if seg.get('_baixa_confianca'):
            prefixo = '🔴 '

        # Linha principal *CHI: com pausas já inseridas
        linhas.append(f'*CHI:\t{prefixo}{timestamp} {texto} .')

        # Linha %com: com info de timing e confiança
        duracao_seg = seg['end'] - seg['start']
        pausa_entre = detectar_pausa_longa(seg, seg_seguinte, limiar_curta, limiar_media, limiar_longa)
        info_partes = [f'dur={duracao_seg:.1f}s']
        if pausa_entre:
            info_partes.append(f'pausa_seguinte={pausa_entre}')
        if seg.get('_baixa_confianca'):
            info_partes.append(f'logprob={seg.get("avg_logprob",0):.2f} — verificar')
        linhas.append(f'%com:\t{", ".join(info_partes)}')
        linhas.append('')

    linhas.append('@End')
    return '\n'.join(linhas)

def gerar_txt_revisao(resultado, codigo, grupo, historia, tarefa,
                       avaliadora, data,
                       limiar_curta=0.5, limiar_media=1.0, limiar_longa=2.0):
    """Gera .txt de revisão com guia de anotação inline."""
    segs = resultado['segments']
    linhas = []

    linhas.append('=' * 65)
    linhas.append('  IDEAL — TRANSCRIÇÃO PARA REVISÃO')
    linhas.append(f'  IDEAL — ESS-IPS / CAIDI')
    linhas.append('=' * 65)
    linhas.append(f'  Participante : {codigo}  |  Grupo: {grupo}')
    linhas.append(f'  História     : {historia}  |  Tarefa: {tarefa}')
    linhas.append(f'  Avaliadora   : {avaliadora}')
    linhas.append(f'  Data         : {data}')
    linhas.append('=' * 65)
    linhas.append('')
    linhas.append('INSTRUÇÕES DE REVISÃO:')
    linhas.append('  1. Ouve o áudio enquanto lês este ficheiro')
    linhas.append('  2. Corrige erros de transcrição')
    linhas.append('  3. Insere anotações CHAT onde necessário:')
    linhas.append('     [/]        → repetição  (ex: "o o [/] o gato")')
    linhas.append('     [//]       → revisão    (ex: "foi para [//] saltou")')
    linhas.append('     (.)        → pausa curta')
    linhas.append('     (..)       → pausa média')
    linhas.append('     (...)      → pausa longa')
    linhas.append('     &-ã        → preenchimento de pausa')
    linhas.append('     [* m:conc] → erro de concordância')
    linhas.append('     [* m:prep] → erro de preposição')
    linhas.append('     [* m:art]  → erro de artigo')
    linhas.append('     [* m:cv]   → erro concordância verbal')
    linhas.append('     [?]        → palavra ininteligível')
    linhas.append('  4. Muda *CHI: para M: nos enunciados não analisáveis')
    linhas.append('  5. Guarda com o nome: ' + f'{codigo}_{historia}_anotado.cha')
    linhas.append('')
    linhas.append('⚠️  NOTA IMPORTANTE — O QUE O WHISPER SUPRIME:')
    linhas.append('    O Whisper foi treinado para produzir texto limpo.')
    linhas.append('    Remove/normaliza activamente: "hm" "ã" "é" "tipo" "pronto"')
    linhas.append('    e outros preenchimentos de pausa (disfluências).')
    linhas.append('    Estes são marcadores clínicos importantes em PDL!')
    linhas.append('    → Locais onde pode ter havido supressão assinalados com ⚡')
    linhas.append('    → Segmentos de baixa confiança (possível normalização) com 🔴')
    linhas.append('    → Ouve SEMPRE o áudio nestes pontos com atenção redobrada')
    linhas.append('')
    linhas.append('⚠️  PAUSAS DETECTADAS entre segmentos assinaladas com →')
    linhas.append('    Pausas DENTRO dos enunciados têm de ser anotadas manualmente.')
    linhas.append('')
    linhas.append('Convenções CHAT a inserir:')
    linhas.append('    &-hm  &-ã  &-e  &-tipo  &-pronto → preenchimentos de pausa')
    linhas.append('    [/]   → repetição imediata')
    linhas.append('    [//]  → revisão / falsa partida')
    linhas.append('    (.)   → pausa curta  (..)  pausa média  (...)  pausa longa')
    linhas.append(f'    [* m:conc] [* m:prep] [* m:art] [* m:cv] → erros gramaticais')
    linhas.append('    [?]   → ininteligível')
    linhas.append('    M:    → início da linha para enunciado não analisável')
    linhas.append('')
    linhas.append(f'Guarda o ficheiro revisto como: {codigo}_{historia}_anotado.cha')
    linhas.append('')
    linhas.append('-' * 65)
    linhas.append('')

    for i, seg in enumerate(segs):
        texto = limpar_texto(seg['text'])
        if not texto:
            continue

        timestamp = ts(seg['start'])
        seg_seguinte = segs[i+1] if i+1 < len(segs) else None
        pausa = detectar_pausa_longa(seg, seg_seguinte, limiar_curta, limiar_media, limiar_longa)
        supressoes = seg.get('_supressoes', [])
        baixa_conf = seg.get('_baixa_confianca', False)

        # Prefixo de aviso se baixa confiança
        prefixo = '🔴 ' if baixa_conf else ''
        linhas.append(f'*CHI:\t{prefixo}{timestamp} {texto} .')

        # Supressões dentro do enunciado
        for sup in supressoes:
            linhas.append(
                f'        ⚡ POSSÍVEL SUPRESSÃO após "{sup["depois_de"]}" '
                f'(gap={sup["gap"]}s) — ouve aqui, inserir &-? ou {sup["tipo"]}')

        # Baixa confiança
        if baixa_conf:
            linhas.append(
                f'        🔴 BAIXA CONFIANÇA — Whisper pode ter normalizado/inventado '
                f'(logprob={seg.get("avg_logprob", 0):.2f}) — verificar áudio')

        # Pausa entre segmentos
        if pausa:
            gap = seg_seguinte['start'] - seg['end']
            linhas.append(f'        → PAUSA {pausa} ({gap:.1f}s antes do próximo enunciado)')

        linhas.append('')

    return '\n'.join(linhas)

def gerar_srt(resultado):
    """Gera .srt para usar no VLC."""
    linhas = []
    for i, seg in enumerate(resultado['segments'], 1):
        linhas += [
            str(i),
            f'{ts_srt(seg["start"])} --> {ts_srt(seg["end"])}',
            seg['text'].strip(),
            ''
        ]
    return '\n'.join(linhas)

print('✅ Funções CHAT carregadas.')

✅ Funções CHAT carregadas.


In [5]:
# ============================================================
# CÉLULA 4 — TRANSCRIÇÃO
# Correr para cada participante
# ============================================================

from google.colab import files
import datetime

# ──────────────────────────────────────────────────────
# 👇 PREENCHE AQUI antes de começar
# ──────────────────────────────────────────────────────
CODIGO       = 'AD003'    # ex: 'CT01'
GRUPO        = 'AD'    # 'CT' / 'PDL' / 'AD'
IDADE        = '23'    # ex: '9;4'
HISTORIA     = 'Gato'    # 'Gato' / 'Cão' / 'Passarinhos' / 'Cabritinhos'
TAREFA       = 'Reconto'    # 'Reconto' / 'Conto'
AVALIADORA   = 'Joana'    # ex: 'Ana Sousa'
DURACAO_MIN  = 2   # duração aproximada em minutos

# ── Limiares de pausa (segundos) ──────────────────────
# Ajusta conforme o grupo:
#   Adultos / CT  → 0.5 / 1.0 / 2.0  (padrão)
#   PDL           → 0.3 / 0.7 / 1.5  (mais sensível)
LIMIAR_PAUSA_CURTA  = 0.5   # (.)   pausa curta
LIMIAR_PAUSA_MEDIA  = 1.0   # (..)  pausa média
LIMIAR_PAUSA_LONGA  = 2.0   # (...) pausa longa

# ── Modo word_timestamps ───────────────────────────────
# True  → pausas inseridas inline no texto, detecção de supressões
#         mais lento (~6-10 min para 20 min de áudio)
# False → mais rápido (~3-5 min), sem pausas inline
WORD_TIMESTAMPS = True

# ──────────────────────────────────────────────────────
DATA = datetime.datetime.now().strftime('%d/%m/%Y')

if not all([CODIGO, GRUPO, HISTORIA, AVALIADORA]):
    print('⚠️  Preenche todos os campos antes de continuar!')
    raise SystemExit('Campos incompletos.')

print('=' * 62)
print(f'  IDEAL — ESS-IPS / CAIDI')
print('=' * 62)
print(f'  Participante : {CODIGO}  |  Grupo: {GRUPO}')
print(f'  História     : {HISTORIA}  |  Tarefa: {TAREFA}')
print(f'  Avaliadora   : {AVALIADORA}  |  Data: {DATA}')
print('=' * 62)

# ── Passo 1: Carregar áudio ────────────────────────────
print('\n📂 Selecciona o ficheiro de áudio (.mp3 / .wav / .m4a):')
uploaded = files.upload()

if not uploaded:
    raise SystemExit('Nenhum ficheiro carregado.')

nome_audio = list(uploaded.keys())[0]
print(f'✅ Áudio carregado: {nome_audio}')

# ── Passo 2: Transcrição faster-whisper ───────────────
tempo_est = '~2-4 min' if WORD_TIMESTAMPS else '~1-2 min'
print(f'\n⏳ A transcrever com faster-whisper large-v3... ({tempo_est} para 20 min de áudio)')
if WORD_TIMESTAMPS:
    print('   word_timestamps=True — pausas inline activas')
else:
    print('   Modo rápido — sem pausas inline')

# faster-whisper devolve um gerador — converter para lista
import math
segs_gen, info = model.transcribe(
    nome_audio,
    language='pt',
    word_timestamps=WORD_TIMESTAMPS,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=300),
)

# Normalizar para o formato dict que o resto do pipeline espera
segmentos = []
for seg in segs_gen:
    words = []
    if WORD_TIMESTAMPS and seg.words:
        for w in seg.words:
            words.append({
                'word':  w.word,
                'start': w.start,
                'end':   w.end,
                'score': w.probability,
            })
    # Converter avg_logprob: faster-whisper usa avg_logprob nativamente
    avg_lp = seg.avg_logprob if hasattr(seg, 'avg_logprob') else -0.5
    no_sp  = seg.no_speech_prob if hasattr(seg, 'no_speech_prob') else 0.0
    segmentos.append({
        'text':          seg.text,
        'start':         seg.start,
        'end':           seg.end,
        'avg_logprob':   avg_lp,
        'no_speech_prob':no_sp,
        'words':         words,
    })

resultado = {'segments': segmentos}

# Passo 3: detectar supressões e baixa confiança
LIMIAR_SUPRESSAO = 0.4
LIMIAR_BAIXA_CONFIANCA = -0.8

def detectar_supressoes(seg):
    supressoes = []
    palavras = seg.get('words', [])
    for i in range(len(palavras) - 1):
        w1 = palavras[i]; w2 = palavras[i+1]
        t1_end   = w1.get('end', 0)
        t2_start = w2.get('start', 0)
        gap = t2_start - t1_end
        if gap >= LIMIAR_SUPRESSAO:
            supressoes.append({
                'depois_de': w1.get('word','').strip(),
                'antes_de':  w2.get('word','').strip(),
                'gap': round(gap, 2),
                'tipo': '(...)' if gap >= LIMIAR_PAUSA_LONGA else '(..)' if gap >= LIMIAR_PAUSA_MEDIA else '(.)'
            })
    return supressoes

def baixa_confianca(seg):
    avg_logprob = seg.get('avg_logprob', -0.5)
    no_speech   = seg.get('no_speech_prob', 0.0)
    return avg_logprob < LIMIAR_BAIXA_CONFIANCA or no_speech > 0.3

n_supressoes = 0
n_baixa_conf = 0
for seg in resultado['segments']:
    seg['_supressoes'] = detectar_supressoes(seg) if WORD_TIMESTAMPS else []
    seg['_baixa_confianca'] = baixa_confianca(seg)
    n_supressoes += len(seg['_supressoes'])
    if seg['_baixa_confianca']: n_baixa_conf += 1

n_segs = len(resultado['segments'])
dur_total = resultado['segments'][-1]['end'] if n_segs else 0
print(f'✅ Transcrição concluída!')
print(f'   {n_segs} segmentos  |  {dur_total/60:.1f} min')
print(f'   Supressões detectadas (gaps ≥{LIMIAR_SUPRESSAO}s): {n_supressoes}')
print(f'   Segmentos baixa confiança: {n_baixa_conf}')
if n_supressoes > 0:
    print(f'   → Locais assinalados no .cha com ⚡')

# ── Passo 3: Gerar ficheiros ───────────────────────────
print('\n📝 A gerar ficheiros CHAT...')

nome_base = f'{CODIGO}_{HISTORIA}'

# .cha — formato CHAT puro para CLAN e Notebook 2
conteudo_cha = gerar_cha(
    resultado, CODIGO, GRUPO, IDADE, HISTORIA, TAREFA,
    AVALIADORA, DATA, DURACAO_MIN,
    LIMIAR_PAUSA_CURTA, LIMIAR_PAUSA_MEDIA, LIMIAR_PAUSA_LONGA
)
nome_cha = f'{nome_base}_base.cha'
with open(nome_cha, 'w', encoding='utf-8') as f:
    f.write(conteudo_cha)

# .txt — versão de revisão com instruções inline
conteudo_txt = gerar_txt_revisao(
    resultado, CODIGO, GRUPO, HISTORIA, TAREFA, AVALIADORA, DATA,
    LIMIAR_PAUSA_CURTA, LIMIAR_PAUSA_MEDIA, LIMIAR_PAUSA_LONGA
)
nome_txt = f'{nome_base}_para_revisao.txt'
with open(nome_txt, 'w', encoding='utf-8') as f:
    f.write(conteudo_txt)

# .srt — para VLC
conteudo_srt = gerar_srt(resultado)
nome_srt = f'{nome_base}.srt'
with open(nome_srt, 'w', encoding='utf-8') as f:
    f.write(conteudo_srt)

# ── Pré-visualização ──────────────────────────────────
print('\n' + '=' * 62)
print('PRÉ-VISUALIZAÇÃO — primeiros 6 enunciados:')
print('=' * 62)
for seg in resultado['segments'][:6]:
    t = limpar_texto(seg['text'])
    print(f'*CHI:\t{ts(seg["start"])} {t} .')
print(f'... (+{max(0, n_segs-6)} enunciados no ficheiro)')

# ── Download ──────────────────────────────────────────
print('\n⬇️  A fazer download...')
files.download(nome_cha)
files.download(nome_txt)
files.download(nome_srt)

print('\n✅ Download concluído! Tens 3 ficheiros:')
print(f'  📄 {nome_cha}     → CHAT base para anotar (envia para Notebook 2 depois)')
print(f'  📄 {nome_txt}  → versão de revisão com instruções')
print(f'  📄 {nome_srt}         → legendas para VLC')
print()
print('PRÓXIMO PASSO:')
print(f'  1. Abre {nome_cha} no Notepad/Word')
print(f'  2. Reproduz o áudio no VLC com o .srt como legenda')
print(f'  3. Revê e anota as convenções CHAT (ver instruções no .txt)')
print(f'  4. Guarda como {CODIGO}_{HISTORIA}_anotado.cha')
print(f'  5. Carrega no Notebook 2 para análise automática')

  IDEAL — ESS-IPS / CAIDI
  Participante : AD003  |  Grupo: AD
  História     : Gato  |  Tarefa: Reconto
  Avaliadora   : Joana  |  Data: 07/04/2026

📂 Selecciona o ficheiro de áudio (.mp3 / .wav / .m4a):


Saving AD003_GATO.mp3 to AD003_GATO.mp3
✅ Áudio carregado: AD003_GATO.mp3

⏳ A transcrever com faster-whisper large-v3... (~2-4 min para 20 min de áudio)
   word_timestamps=True — pausas inline activas
✅ Transcrição concluída!
   6 segmentos  |  1.4 min
   Supressões detectadas (gaps ≥0.4s): 9
   Segmentos baixa confiança: 0
   → Locais assinalados no .cha com ⚡

📝 A gerar ficheiros CHAT...

PRÉ-VISUALIZAÇÃO — primeiros 6 enunciados:
*CHI:	[00:00] ok, então, era uma vez um gato que estava ao pé do rio e encontrou uma borboleta amarela e saltou para apanhar .
*CHI:	[00:17] depois estava um rapaz que estava ao pé do rio para ir pescar e depois viu o gato e ele caiu para o arbusto .
*CHI:	[00:39] depois ele tinha uma bola vermelha e quando foi tentar salvar o gato, a bola caiu para o rio e ele ficou triste .
*CHI:	[01:05] e depois tentou ir apanhar a bola com a cana de pesca e não reparou que o gato tinha ido buscar os peixes que ele tinha buscado .
*CHI:	[01:20] mas ele estava feliz porq

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download concluído! Tens 3 ficheiros:
  📄 AD003_Gato_base.cha     → CHAT base para anotar (envia para Notebook 2 depois)
  📄 AD003_Gato_para_revisao.txt  → versão de revisão com instruções
  📄 AD003_Gato.srt         → legendas para VLC

PRÓXIMO PASSO:
  1. Abre AD003_Gato_base.cha no Notepad/Word
  2. Reproduz o áudio no VLC com o .srt como legenda
  3. Revê e anota as convenções CHAT (ver instruções no .txt)
  4. Guarda como AD003_Gato_anotado.cha
  5. Carrega no Notebook 2 para análise automática


---

## Estrutura do ficheiro `.cha` gerado

```
@Begin
@Languages:   pt
@Participants: CHI CT01 Target_Child, INV Investigadora Investigator
@ID:          pt|IDEAL|CHI|9;4||CT||Target_Child|
@Date:        15/03/2026
@Comment:     Historia=Gato Tarefa=Reconto Avaliadora=Ana Sousa
@Comment:     ⚠️ Transcrição automática — rever e completar

*CHI:  [00:09] saltou para apanhá-la enquanto chegava o menino .
%com:  dur=4.2s

*CHI:  [00:15] o gato ficou muito zangado .
%com:  dur=2.1s, pausa_seguinte=(..) (1.8s antes do próximo enunciado)

@End
```

## Depois de anotar, o ficheiro fica assim:

```
*CHI:  [00:09] saltou para apanhá [/] apanhá-la enquanto (.) chegava o menino .
%com:  dur=4.2s

*CHI:  [00:15] o gato ficou (.) ficou [//] ficou muito zangado .
%com:  dur=2.1s, pausa_seguinte=(..) (1.8s antes do próximo enunciado)
```

## Abrir no VLC com legendas
1. Abre o áudio no VLC
2. **Legendas → Adicionar ficheiro de legendas** → selecciona o `.srt`
3. Vês o texto sincronizado enquanto ouves

## Nome dos ficheiros para processamento em lote

Os ficheiros de áudio devem seguir este formato:
```
CODIGO_HISTORIA.extensão

CT01_Gato.mp3
CT02_Cabritinhos.wav
PDL03_Passarinhos.m4a
AD01_Cão.mp3
```
Histórias válidas: `Gato` `Cão` `Passarinhos` `Cabritinhos` (maiúscula + acentos)


In [6]:
# ============================================================
# CÉLULA 5 — LOTE (vários participantes de uma vez)
#
# Os ficheiros devem chamar-se: CODIGO_HISTORIA.extensão
# Exemplos: CT01_Gato.mp3  PDL03_Passarinhos.wav  AD01_Cão.m4a
#
# O notebook lê o código e a história do nome do ficheiro.
# Preenche apenas os campos comuns a todos os participantes.
# ============================================================

import datetime
import os
import re
from google.colab import files

# ──────────────────────────────────────────────────────
# 👇 PREENCHE AQUI — campos comuns a todos
# ──────────────────────────────────────────────────────
GRUPO_LOTE      = 'AD'    # 'CT' / 'PDL' / 'AD'
TAREFA_LOTE     = 'Reconto'    # 'Reconto' / 'Conto'
AVALIADORA_LOTE = 'Mariana'    # ex: 'Ana Sousa'

# Limiares de pausa (mesmos da Célula 4)
LIMIAR_CURTA_L  = 0.5
LIMIAR_MEDIA_L  = 1.0
LIMIAR_LONGA_L  = 2.0
WORD_TS_LOTE    = True   # False = mais rápido

# ──────────────────────────────────────────────────────
HISTORIAS_VALIDAS = ['Gato', 'Cão', 'Passarinhos', 'Cabritinhos']
DATA_LOTE = datetime.datetime.now().strftime('%d/%m/%Y')

if not all([GRUPO_LOTE, TAREFA_LOTE, AVALIADORA_LOTE]):
    print('⚠️  Preenche GRUPO_LOTE, TAREFA_LOTE e AVALIADORA_LOTE antes de continuar!')
    raise SystemExit('Campos incompletos.')

def extrair_codigo_historia(nome_ficheiro):
    """
    Extrai código e história do nome do ficheiro.
    CT01_Gato.mp3 → ('CT01', 'Gato')
    PDL03_Passarinhos.wav → ('PDL03', 'Passarinhos')
    """
    nome_base = os.path.splitext(nome_ficheiro)[0]  # remove extensão
    partes = nome_base.split('_', 1)
    if len(partes) != 2:
        return None, None
    codigo = partes[0]
    historia = partes[1]
    if historia not in HISTORIAS_VALIDAS:
        return codigo, None
    return codigo, historia

def transcrever_participante(nome_audio, codigo, historia, grupo, tarefa,
                              avaliadora, data, word_ts,
                              lc, lm, ll):
    """Transcreve um ficheiro com WhisperX e gera os outputs CHAT."""
    print(f'\n⏳ [{codigo} — {historia}] A transcrever...')

    import math
    segs_gen, info = model.transcribe(
        nome_audio,
        language='pt',
        word_timestamps=word_ts,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
    )
    segmentos = []
    for seg in segs_gen:
        words = []
        if word_ts and seg.words:
            for w in seg.words:
                words.append({'word':w.word,'start':w.start,'end':w.end,'score':w.probability})
        segmentos.append({
            'text': seg.text, 'start': seg.start, 'end': seg.end,
            'avg_logprob': seg.avg_logprob if hasattr(seg,'avg_logprob') else -0.5,
            'no_speech_prob': seg.no_speech_prob if hasattr(seg,'no_speech_prob') else 0.0,
            'words': words,
        })
    res = {'segments': segmentos}
    for seg in res['segments']:
        seg['_supressoes'] = detectar_supressoes(seg) if word_ts else []
        seg['_baixa_confianca'] = baixa_confianca(seg)

    n_segs = len(res['segments'])
    dur = res['segments'][-1]['end'] / 60 if n_segs else 0

    nome_base = f'{codigo}_{historia}'

    cha = gerar_cha(res, codigo, grupo, '', historia, tarefa,
                    avaliadora, data, round(dur, 1), lc, lm, ll)
    txt = gerar_txt_revisao(res, codigo, grupo, historia, tarefa,
                             avaliadora, data, lc, lm, ll)
    srt = gerar_srt(res)

    with open(f'{nome_base}_base.cha', 'w', encoding='utf-8') as f: f.write(cha)
    with open(f'{nome_base}_para_revisao.txt', 'w', encoding='utf-8') as f: f.write(txt)
    with open(f'{nome_base}.srt', 'w', encoding='utf-8') as f: f.write(srt)

    print(f'  ✅ {n_segs} segmentos | {dur:.1f} min')
    return nome_base

# ── Carregar ficheiros ─────────────────────────────────
print('📂 Selecciona TODOS os ficheiros de áudio:')
print('   Formato obrigatório: CODIGO_HISTORIA.extensão')
print('   Ex: CT01_Gato.mp3  PDL03_Passarinhos.wav  AD01_Cão.m4a')
uploaded_lote = files.upload()

if not uploaded_lote:
    raise SystemExit('Nenhum ficheiro carregado.')

# ── Validar nomes ──────────────────────────────────────
validos = []
invalidos = []
for nome in uploaded_lote.keys():
    codigo, historia = extrair_codigo_historia(nome)
    if codigo and historia:
        validos.append((nome, codigo, historia))
    else:
        invalidos.append(nome)

if invalidos:
    print(f'\n⚠️  Ficheiros com nome inválido (serão ignorados):')
    for n in invalidos:
        _, h = extrair_codigo_historia(n)
        if h is None:
            print(f'   {n} → história não reconhecida. Válidas: {HISTORIAS_VALIDAS}')
        else:
            print(f'   {n} → formato incorrecto. Usa: CODIGO_HISTORIA.extensão')

if not validos:
    raise SystemExit('Nenhum ficheiro com nome válido.')

print(f'\n✅ {len(validos)} ficheiro(s) válido(s) para processar:')
for nome, cod, hist in validos:
    print(f'   {nome} → {cod} / {hist}')

# ── Processar em lote ──────────────────────────────────
print(f'\n🚀 A iniciar processamento em lote...')
print(f'   Modo: {"word_timestamps" if WORD_TS_LOTE else "rápido"}')
print(f'   Tempo estimado: ~{len(validos) * (8 if WORD_TS_LOTE else 4)} min')
print(f'   Não feches o browser!\n')

resultados_lote = []
for i, (nome_audio, codigo, historia) in enumerate(validos, 1):
    print(f'[{i}/{len(validos)}] {nome_audio}')
    try:
        nome_base = transcrever_participante(
            nome_audio, codigo, historia,
            GRUPO_LOTE, TAREFA_LOTE, AVALIADORA_LOTE, DATA_LOTE,
            WORD_TS_LOTE, LIMIAR_CURTA_L, LIMIAR_MEDIA_L, LIMIAR_LONGA_L
        )
        resultados_lote.append({'nome': nome_audio, 'base': nome_base, 'ok': True})
    except Exception as e:
        print(f'  ❌ Erro: {e}')
        resultados_lote.append({'nome': nome_audio, 'ok': False, 'erro': str(e)})

# ── Resumo ─────────────────────────────────────────────
print('\n' + '=' * 50)
print('RESUMO DO LOTE:')
print('=' * 50)
for r in resultados_lote:
    estado = '✅' if r['ok'] else '❌'
    print(f'  {estado} {r["nome"]}')

# ── Download de todos os ficheiros ─────────────────────
print('\n⬇️  A fazer download de todos os ficheiros...')
for r in resultados_lote:
    if r['ok']:
        files.download(f'{r["base"]}_base.cha')
        files.download(f'{r["base"]}_para_revisao.txt')
        files.download(f'{r["base"]}.srt')

print('\n✅ Download concluído!')
print('   Para cada participante tens 3 ficheiros:')
print('   _base.cha        → anotar e entregar ao Notebook 2')
print('   _para_revisao.txt → guia de revisão com avisos')
print('   .srt              → legendas para VLC')

📂 Selecciona TODOS os ficheiros de áudio:
   Formato obrigatório: CODIGO_HISTORIA.extensão
   Ex: CT01_Gato.mp3  PDL03_Passarinhos.wav  AD01_Cão.m4a


KeyboardInterrupt: 